Dans ce notebook on va séparer en train et test nos données avant analyse préalable de la base pour éviter toute forme de data leakage. 

In [1]:
import os 
import pandas as pd 
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go

In [2]:
print(os.getcwd())
os.chdir("../")
os.getcwd()

c:\Users\leoco\Documents\cours\M2_MOSEF\ML_theory\land_value_prediction\notebooks


'c:\\Users\\leoco\\Documents\\cours\\M2_MOSEF\\ML_theory\\land_value_prediction'

In [3]:
from src.land_value_prediction.train_test_split.analysis_functions import comparer_train_test, identifier_differences_significatives

In [4]:
idf_vf_full = pd.read_parquet("data/processed/idf_vf_full.parquet")
idf_vf_full

,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,nb_restauration_2000m,nb_sante_2000m,nb_loisirs_2000m,nb_services_2000m,nb_transport_lourd_2000m,nb_bus_2000m,nb_transport_autre_2000m,nb_total_2000m,diversite_2000m,densite_rel_2000m
0,2021-01-01,Vente,169500.0,28.0,ALL HOCHE,4440,92130.0,92040,Issy-les-Moulineaux,92,...,14,5,335,6,32,351,9,825,9,1.210341
1,2021-01-02,Vente,185000.0,20.0,AV GAL LECLERC,0360,95250.0,95051,Beauchamp,95,...,13,1,160,4,1,113,1,329,9,-0.372779
2,2021-01-02,Vente,415000.0,109.0,RUE DES COURLIS,1570,95100.0,95018,Argenteuil,95,...,16,3,96,4,1,211,0,366,8,-0.254683
3,2021-01-02,Vente,415000.0,109.0,RUE DES COURLIS,1570,95100.0,95018,Argenteuil,95,...,16,3,96,4,1,211,0,366,8,-0.254683
4,2021-01-04,Vente,255000.0,5.0,CHE DU MARCREUX,6115,93300.0,93001,Aubervilliers,93,...,5,6,251,5,19,217,2,557,9,0.354946
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
634231,2025-06-30,Vente,283000.0,4.0,RUE FRANCOIS MITTERRAND,0347,77380.0,77122,Combs-la-Ville,77,...,4,2,94,2,0,63,0,170,7,-1.089169
634232,2025-06-30,Vente,144434.0,1.0,RUE JEAN DUSSART,1450,91390.0,91434,Morsang-sur-Orge,91,...,8,1,213,6,12,203,0,488,8,-0.129552
634233,2025-06-30,Vente,300000.0,32.0,RUE DU VAL ANDRE,0150,78560.0,78502,Le Port-Marly,78,...,2,2,326,3,3,190,2,600,9,0.208427
634234,2025-06-30,Vente,476000.0,1.0,IMP DE LA FORET,0322,78450.0,78674,Villepreux,78,...,1,1,64,2,1,86,1,181,9,-1.055975


On va créer 3 échantillons :  
- échantillons train et test sur 2021-2024  
- échantillon temporel out_of_sample (2025-S1)  

Séparation train-test et échantillon out of sample :

In [5]:
train_test = idf_vf_full[idf_vf_full['annee'] < 2025].copy()
out_of_sample_df = idf_vf_full[idf_vf_full['annee'] >= 2025].copy()

print(f"Taille du train-test: {len(train_test)} ({len(train_test)/len(idf_vf_full)*100:.1f}%)")
print(f"Taille du out of sample: {len(out_of_sample_df)} ({len(out_of_sample_df)/len(idf_vf_full)*100:.1f}%)")
print(f"\nPériode train-test: {train_test['date_mutation'].min()} à {train_test['date_mutation'].max()}")
print(f"Période out of sample: {out_of_sample_df['date_mutation'].min()} à {out_of_sample_df['date_mutation'].max()}")

print("\nStatistiques prix_m2 - Train-test:")
train_test['prix_m2'].describe().round(2)

Taille du train-test: 586468 (92.5%)
Taille du out of sample: 47768 (7.5%)

Période train-test: 2021-01-01 00:00:00 à 2024-12-31 00:00:00
Période out of sample: 2025-01-02 00:00:00 à 2025-06-30 00:00:00

Statistiques prix_m2 - Train-test:


count    586468.00
mean       5843.14
std        3198.67
min        1500.00
25%        3313.95
50%        4769.23
75%        8002.77
max       15000.00
Name: prix_m2, dtype: float64

In [6]:
print("\nStatistiques prix_m2 - Out of sample:")
out_of_sample_df['prix_m2'].describe().round(2)


Statistiques prix_m2 - Out of sample:


count    47768.00
mean      5655.15
std       3065.99
min       1500.00
25%       3235.29
50%       4666.67
75%       7727.27
max      15000.00
Name: prix_m2, dtype: float64

Séparer train et test :

In [7]:
# Créer des déciles sur la variable cible pour stratifier le split
train_test['prix_m2_bins'] = pd.qcut(train_test['prix_m2'], q=10, labels=False, duplicates='drop')

# Split train/test avec stratification
train, test = train_test_split(
    train_test, 
    test_size=0.2, 
    shuffle=True,
    random_state=42, 
    stratify=train_test['prix_m2_bins']
)

# Supprimer la colonne temporaire de bins
train = train.drop('prix_m2_bins', axis=1)
test = test.drop('prix_m2_bins', axis=1)

print(f"Taille du train set: {len(train)} ({len(train)/len(train_test)*100:.1f}%)")
print(f"Taille du test set: {len(test)} ({len(test)/len(train_test)*100:.1f}%)")

print("\nStatistiques prix_m2 - Train:")
train['prix_m2'].describe()

Taille du train set: 469174 (80.0%)
Taille du test set: 117294 (20.0%)

Statistiques prix_m2 - Train:


count    469174.000000
mean       5843.766070
std        3199.308239
min        1500.000000
25%        3314.446970
50%        4769.230769
75%        8007.162102
max       15000.000000
Name: prix_m2, dtype: float64

In [8]:
print("\nStatistiques prix_m2 - Test:")
test['prix_m2'].describe()


Statistiques prix_m2 - Test:


count    117294.000000
mean       5840.611200
std        3196.141254
min        1500.000000
25%        3312.105005
50%        4769.230769
75%        8000.000000
max       15000.000000
Name: prix_m2, dtype: float64

In [9]:
comparison = comparer_train_test(train, test)
comparison

,type,train_mean,test_mean,diff_mean,train_median,test_median,diff_median,train_std,test_std,diff_std,train_min,test_min,train_max,test_max,diff_mean_pct,diff_median_pct,diff_std_pct
variable,,,,,,,,,,,,,,,,,
date_mutation,categorical,<NA>,<NA>,<NA>,2021-07-30 00:00:00,2021-06-30 00:00:00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
nature_mutation,categorical,<NA>,<NA>,<NA>,Vente,Vente,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
valeur_fonciere,numeric,384528.335128,384083.339996,444.995132,300000.0,300000.0,0.0,311356.931993,311537.670756,-180.738763,15000.0,15000.0,8150000.0,8929250.0,0.115859,0.0,-0.058015
adresse_numero,numeric,127.770616,129.153807,-1.383191,19.0,19.0,0.0,851.913207,858.257126,-6.343919,1.0,1.0,9999.0,9248.0,-1.070965,0.0,-0.739163
adresse_nom_voie,categorical,<NA>,<NA>,<NA>,RUE DE PARIS,RUE DE PARIS,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
nb_bus_2000m,numeric,195.508568,195.123868,0.3847,176.0,175.0,1.0,126.461117,126.61321,-0.152093,0,0,536,535,0.197157,0.571429,-0.120124
nb_transport_autre_2000m,numeric,6.603424,6.584446,0.018978,1.0,1.0,0.0,11.580529,11.566321,0.014207,0,0,63,63,0.288231,0.0,0.122834
nb_total_2000m,numeric,483.17849,482.121754,1.056736,419.0,417.0,2.0,325.579092,325.972186,-0.393094,0,0,1245,1243,0.219185,0.479616,-0.120591


In [10]:
warnings = identifier_differences_significatives(comparison, threshold_mean=0.1, threshold_std=0.2)

# Afficher les alertes
if warnings['mean']:
    print("⚠️ Variables avec différences de moyenne > 10%:")
    for w in warnings['mean']:
        print(f"  - {w['variable']}: {w['diff_pct']:.1f}%")

if warnings['std']:
    print("\n⚠️ Variables avec différences d'écart-type > 20%:")
    for w in warnings['std']:
        print(f"  - {w['variable']}: {w['diff_pct']:.1f}%")

if warnings['distribution']:
    print("\n⚠️ Variables avec plage de valeurs différentes:")
    for w in warnings['distribution']:
        print(f"  - {w['variable']}: train={w['train_range']}, test={w['test_range']}")

⚠️ Variables avec différences de moyenne > 10%:
  - densite_rel_500m: -125.0%
  - densite_rel_1000m: -125.0%
  - densite_rel_2000m: -125.0%

⚠️ Variables avec différences d'écart-type > 20%:
  - surface_terrain: -39.5%

⚠️ Variables avec plage de valeurs différentes:
  - valeur_fonciere: train=[15000.0, 8150000.0], test=[15000.0, 8929250.0]
  - surface_terrain: train=[1.0, 181873.0], test=[1.0, 300000.0]
  - ecart_prix_median_pct: train=[-88.94179894179894, 658.8559093362117], test=[-88.19117647058823, 670.2792308259045]
  - commune_densite_pop: train=[3.1712473572938684, 39957.22070844687], test=[3.012684989429175, 39957.22070844687]
  - nb_transport_lourd_500m: train=[0, 15], test=[0, 16]
  - dist_bus: train=[2.330614456549587, 3894.2852258400026], test=[3.6189614816282707, 3982.30517248265]
  - densite_rel_500m: train=[-1.7356487654917367, 6.847022349127552], test=[-1.7356487654917367, 6.89441211425111]
  - nb_bus_1000m: train=[0, 176], test=[0, 178]


In [11]:
fig = go.Figure()

# Ajouter les distributions
fig.add_trace(go.Box(
    y=train['prix_m2'],
    name='Train',
    boxmean='sd',
    marker_color='lightblue'
))

fig.add_trace(go.Box(
    y=test['prix_m2'],
    name='Test',
    boxmean='sd',
    marker_color='lightgreen'
))

fig.add_trace(go.Box(
    y=out_of_sample_df['prix_m2'],
    name='Out of Sample (2025)',
    boxmean='sd',
    marker_color='lightcoral'
))

# Mise en forme
fig.update_layout(
    title='Comparaison de la distribution de prix_m2 entre les échantillons',
    yaxis_title='Prix au m² (€)',
    showlegend=True,
    height=600,
    template='plotly_white'
)

fig.show()

In [12]:
train.to_parquet("data/processed/train_test_out_sample_split/idf_vf_train.parquet", index=False)
test.to_parquet("data/processed/train_test_out_sample_split/idf_vf_test.parquet", index=False)
out_of_sample_df.to_parquet("data/processed/train_test_out_sample_split/idf_vf_out_of_sample.parquet", index=False)